# MLB Moneyline Development and QA Notebook

This notebook is the development layer for the MLB moneyline model. It performs data QA, EDA, feature engineering validation, hyperparameter tuning, calibration checks, and a simple betting simulation.

Production scripts live under `scripts/`. Use this notebook to inspect whether those scripts are producing trustworthy data and whether the model is actually adding value over the betting market.


## 0. Setup

Run the production scripts first when starting from scratch:

```bash
python scripts/00_init_db.py
python scripts/02_fetch_mlb_games.py --days-back 730 --days-forward 14
python scripts/01_fetch_odds.py --sport baseball_mlb --regions us --markets h2h,spreads,totals
python scripts/03_build_features.py
```


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score, accuracy_score

from mlb_betting.config import get_settings
from mlb_betting.db import connect, init_db, read_sql
from mlb_betting.feature_engineering import (
    load_mlb_games,
    load_latest_odds_consensus,
    build_game_feature_frame,
    get_model_feature_columns,
    save_features,
)
from mlb_betting.data_validation import summarize_missingness, validate_no_obvious_leakage, basic_game_checks
from mlb_betting.modeling import tune_moneyline_model, save_model_bundle
from mlb_betting.betting_math import expected_value_per_unit, profit_if_win

settings = get_settings()
init_db(settings.odds_db_path)
settings


## 1. Load raw tables

The database contains both raw snapshots and normalized tables. Start with a simple inventory check.


In [ ]:
with connect(settings.odds_db_path) as conn:
    tables = {
        'odds_events': read_sql(conn, 'SELECT COUNT(*) AS n FROM odds_events')['n'].iloc[0],
        'odds_snapshots': read_sql(conn, 'SELECT COUNT(*) AS n FROM odds_snapshots')['n'].iloc[0],
        'mlb_games': read_sql(conn, 'SELECT COUNT(*) AS n FROM mlb_games')['n'].iloc[0],
        'completed_mlb_games': read_sql(conn, 'SELECT COUNT(*) AS n FROM mlb_games WHERE target_home_win IS NOT NULL')['n'].iloc[0],
    }
tables


In [ ]:
with connect(settings.odds_db_path) as conn:
    games = load_mlb_games(conn)
    odds_consensus = load_latest_odds_consensus(conn)

print(basic_game_checks(games))
games.head()


## 2. EDA: MLB game data

We check date coverage, target balance, scores, and missing fields. The target is `target_home_win`.


In [ ]:
completed_games = games[games['target_home_win'].notna()].copy()
completed_games['game_datetime_utc'] = pd.to_datetime(completed_games['game_datetime_utc'], utc=True)
completed_games['season'] = completed_games['season'].astype('Int64')

eda_summary = {
    'rows': len(games),
    'completed_rows': len(completed_games),
    'min_completed_date': str(completed_games['official_date'].min()) if len(completed_games) else None,
    'max_completed_date': str(completed_games['official_date'].max()) if len(completed_games) else None,
    'home_win_rate': completed_games['target_home_win'].mean() if len(completed_games) else None,
    'avg_total_runs': completed_games['total_runs'].mean() if len(completed_games) else None,
}
eda_summary


In [ ]:
summarize_missingness(games).head(25)


In [ ]:
if len(completed_games):
    completed_games.groupby('season').agg(
        games=('game_pk', 'count'),
        home_win_rate=('target_home_win', 'mean'),
        avg_home_runs=('home_score', 'mean'),
        avg_away_runs=('away_score', 'mean'),
        avg_total_runs=('total_runs', 'mean'),
    )


In [ ]:
if len(completed_games):
    ax = completed_games['target_home_win'].value_counts(normalize=True).sort_index().plot(kind='bar')
    ax.set_title('Home Win Target Distribution')
    ax.set_xlabel('target_home_win')
    ax.set_ylabel('Share')
    plt.show()


In [ ]:
if len(completed_games):
    ax = completed_games['total_runs'].hist(bins=30)
    ax.set_title('Distribution of Total Runs')
    ax.set_xlabel('Total runs')
    ax.set_ylabel('Games')
    plt.show()


## 3. Odds EDA

If you have not gathered odds snapshots yet, this section will be sparse. As the database grows, this becomes the foundation for line movement, closing-line value, and model-vs-market testing.


In [ ]:
odds_consensus.head()


In [ ]:
if len(odds_consensus):
    display(odds_consensus[['event_date','odds_away_team','odds_home_team','home_moneyline_median','away_moneyline_median','market_home_no_vig_prob','market_vig','book_count_h2h_home']].head(20))
    print(odds_consensus[['market_home_no_vig_prob','market_vig','book_count_h2h_home']].describe())
else:
    print('No odds consensus rows yet. Run scripts/01_fetch_odds.py to populate odds snapshots.')


## 4. Feature engineering

Rolling features are shifted by one game, so a team's current game result is not used to predict itself. This is the most important leakage-control step in the starter model.


In [ ]:
features = build_game_feature_frame(games, odds_consensus=odds_consensus, include_future=True)
feature_path = settings.data_dir / 'processed' / 'mlb_game_features.parquet'
save_features(features, feature_path)
features.shape, feature_path


In [ ]:
features.head()


In [ ]:
feature_cols = get_model_feature_columns(features)
validate_no_obvious_leakage(feature_cols)
print(f'Number of model features: {len(feature_cols)}')
feature_cols[:50]


In [ ]:
summarize_missingness(features[feature_cols]).head(30)


### Feature sanity checks

These checks help detect leakage or joins that silently failed. Early-season rows should naturally have more missing rolling features because there is limited prior history.


In [ ]:
completed_features = features[features['target_home_win'].notna()].copy()
if len(completed_features):
    check_cols = [c for c in feature_cols if 'last10' in c or 'market' in c][:20]
    display(completed_features[['official_date','away_team_name','home_team_name','target_home_win'] + check_cols].head(10))


In [ ]:
if len(completed_features):
    numeric_preview = completed_features[feature_cols].select_dtypes(include='number')
    corr = numeric_preview.corr(numeric_only=True)['diff_win_last10'].sort_values(ascending=False) if 'diff_win_last10' in numeric_preview else None
    if corr is not None:
        display(corr.head(15))
        display(corr.tail(15))


## 5. Time-based train/test split and model tuning

For sports betting, use walk-forward or time-based splits. Random splits are optimistic because team form, injuries, market behavior, and roster strength are time dependent.


In [ ]:
trainable = completed_features.sort_values(['game_datetime_utc','game_pk']).reset_index(drop=True).copy()
print('Trainable rows:', len(trainable))
print('Date range:', trainable['official_date'].min(), 'to', trainable['official_date'].max())


In [ ]:
if len(trainable) >= 100:
    result = tune_moneyline_model(
        trainable,
        feature_cols=feature_cols,
        holdout_days=45,
        tune=True,
        calibrate=True,
    )
    print('Run ID:', result['run_id'])
    print('Selected model:', result['model_name'])
    print('Final metrics:', json.dumps(result['final_metrics'], indent=2))
    pd.DataFrame(result['search_results'])[['model_name','cv_log_loss','holdout_metrics','best_params']]
else:
    result = None
    print('Not enough completed games to tune yet. Fetch more history with scripts/02_fetch_mlb_games.py --days-back 730')


## 6. Calibration analysis

A betting model must be calibrated. A 60% predicted probability should win close to 60% over enough games.


In [ ]:
if result is not None:
    holdout = result['holdout_predictions'].copy()
    prob_true, prob_pred = calibration_curve(holdout['target_home_win'], holdout['model_home_win_prob'], n_bins=8, strategy='quantile')
    plt.plot(prob_pred, prob_true, marker='o')
    plt.plot([0,1], [0,1], linestyle='--')
    plt.title('Calibration Curve - Holdout')
    plt.xlabel('Mean predicted probability')
    plt.ylabel('Observed win rate')
    plt.show()
    display(holdout.head())


In [ ]:
if result is not None:
    holdout = result['holdout_predictions'].copy()
    holdout['prob_bucket'] = pd.qcut(holdout['model_home_win_prob'], q=8, duplicates='drop')
    display(holdout.groupby('prob_bucket').agg(
        games=('game_pk','count'),
        avg_pred=('model_home_win_prob','mean'),
        actual_win_rate=('target_home_win','mean'),
    ))


## 7. Market comparison

If odds snapshots exist for holdout games, compare model probabilities to no-vig market probabilities. This is a better benchmark than raw accuracy.


In [ ]:
if result is not None:
    holdout_full = result['holdout_predictions'].merge(
        features[['game_pk','market_home_no_vig_prob','home_moneyline_median','away_moneyline_median']],
        on='game_pk',
        how='left'
    )
    market_rows = holdout_full['market_home_no_vig_prob'].notna().sum()
    print('Holdout rows with market odds:', market_rows)
    if market_rows:
        y = holdout_full.loc[holdout_full['market_home_no_vig_prob'].notna(), 'target_home_win']
        m = holdout_full.loc[holdout_full['market_home_no_vig_prob'].notna(), 'market_home_no_vig_prob']
        print({
            'market_log_loss': log_loss(y, m),
            'market_brier': brier_score_loss(y, m),
            'model_same_rows_log_loss': log_loss(y, holdout_full.loc[holdout_full['market_home_no_vig_prob'].notna(), 'model_home_win_prob']),
            'model_same_rows_brier': brier_score_loss(y, holdout_full.loc[holdout_full['market_home_no_vig_prob'].notna(), 'model_home_win_prob']),
        })


## 8. Feature importance

Permutation importance is model-agnostic and works with the final tuned pipeline. Use it on the holdout period only.


In [ ]:
if result is not None and len(result['test_index']):
    test_df = trainable.loc[result['test_index']].copy() if max(result['test_index']) < len(trainable) else trainable.iloc[-len(result['holdout_predictions']):].copy()
    X_test = test_df[feature_cols]
    y_test = test_df['target_home_win'].astype(int)
    estimator = result['estimator']
    perm = permutation_importance(estimator, X_test, y_test, scoring='neg_log_loss', n_repeats=5, random_state=42, n_jobs=-1)
    imp = pd.DataFrame({
        'feature': feature_cols,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std,
    }).sort_values('importance_mean', ascending=False)
    display(imp.head(30))
    ax = imp.head(20).sort_values('importance_mean').plot(kind='barh', x='feature', y='importance_mean', legend=False)
    ax.set_title('Permutation Importance - Holdout')
    ax.set_xlabel('Increase in negative log loss when shuffled')
    plt.show()


## 9. Simple betting simulation

This is intentionally basic. It bets only when model probability exceeds no-vig market probability by an edge threshold. Use it for QA, not as final bankroll management.


In [ ]:
def simulate_moneyline_bets(df, min_edge=0.02, stake=1.0):
    rows = []
    for _, row in df.dropna(subset=['market_home_no_vig_prob','home_moneyline_median','away_moneyline_median']).iterrows():
        model_home = row['model_home_win_prob']
        market_home = row['market_home_no_vig_prob']
        market_away = 1 - market_home
        home_edge = model_home - market_home
        away_edge = (1 - model_home) - market_away
        if home_edge >= min_edge and home_edge >= away_edge:
            won = row['target_home_win'] == 1
            price = row['home_moneyline_median']
            profit = profit_if_win(price, stake) if won else -stake
            rows.append({'game_pk': row['game_pk'], 'side': 'home', 'price': price, 'edge': home_edge, 'won': won, 'profit': profit})
        elif away_edge >= min_edge:
            won = row['target_home_win'] == 0
            price = row['away_moneyline_median']
            profit = profit_if_win(price, stake) if won else -stake
            rows.append({'game_pk': row['game_pk'], 'side': 'away', 'price': price, 'edge': away_edge, 'won': won, 'profit': profit})
    bets = pd.DataFrame(rows)
    if bets.empty:
        return bets, {'bets': 0}
    summary = {
        'bets': len(bets),
        'win_rate': bets['won'].mean(),
        'profit_units': bets['profit'].sum(),
        'roi': bets['profit'].sum() / (stake * len(bets)),
        'avg_edge': bets['edge'].mean(),
    }
    return bets, summary

if result is not None:
    holdout_market = result['holdout_predictions'].merge(
        features[['game_pk','market_home_no_vig_prob','home_moneyline_median','away_moneyline_median']],
        on='game_pk', how='left'
    )
    for edge in [0.01, 0.02, 0.03, 0.05]:
        bets, summary = simulate_moneyline_bets(holdout_market, min_edge=edge)
        print(edge, summary)


## 10. Save model bundle

Once QA looks acceptable, save the tuned model artifact. The production scoring script can use the latest `.joblib` bundle.


In [ ]:
if result is not None:
    paths = save_model_bundle(result, settings.model_dir)
    {k: str(v) for k, v in paths.items()}


## 11. Next feature upgrades

The starter features are intentionally simple and leakage-safe. The next upgrades should be added one at a time and validated with walk-forward testing:

1. Starting pitcher quality: season-to-date ERA, FIP/xFIP proxy, K-BB%, pitch count trend, rest days.
2. Bullpen workload: relief innings/pitches over last 1, 3, 5 days.
3. Team batting splits: rolling wOBA/OPS/ISO by opposing starter handedness.
4. Park and weather: run environment, wind direction/speed, temperature.
5. Line movement: open/current/close delta and bookmaker dispersion.
6. Injury and lineup strength: projected lineup when available.

Add each group behind a clear feature namespace and re-run this notebook before promoting it into production scripts.
